# Fraud Detection Machine Learning Pipeline

This notebook demonstrates the end-to-end machine learning workflow using curated Gold Layer data stored in AWS S3.

The objective is to train a fraud detection model capable of identifying suspicious financial transactions using engineered behavioral and transactional features.

In [1]:
# Imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score)
import pandas as pd
import numpy as np
import joblib
import os
from dotenv import load_dotenv

In [2]:
# Creating Spark session

spark = SparkSession.builder \
    .appName("FraudDetectionML") \
    .config(
        "spark.jars",
        "/home/jovyan/work/jars/hadoop-aws-3.3.4.jar,/home/jovyan/work/jars/aws-java-sdk-bundle-1.12.262.jar"
    ) \
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    ) \
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    ) \
    .getOrCreate()

In [3]:
# Configuring AWS credentials

load_dotenv("/home/jovyan/work/.env")

AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")

spark._jsc.hadoopConfiguration().set(
    "fs.s3a.access.key",
    AWS_ACCESS_KEY
)

spark._jsc.hadoopConfiguration().set(
    "fs.s3a.secret.key",
    AWS_SECRET_KEY
)

spark._jsc.hadoopConfiguration().set(
    "fs.s3a.endpoint",
    "s3.amazonaws.com"
)

In [4]:
# Reading Gold layer from S3

df_gold = spark.read.parquet(
    "s3a://fraud-detection-data-lake-200702211381/gold/fraud_features/"
)

print(f"Total Gold rows: {df_gold.count():,}")

Total Gold rows: 1,316,675


In [5]:
# Selecting features for fraud detection model

selected_columns = [
    "amt",
    "city_pop",
    "lat",
    "long",
    "merch_lat",
    "merch_long",
    "unix_time",
    "transaction_hour",
    "is_weekend",
    "customer_age",
    "high_amount_flag",
    "night_transaction_flag",
    "customer_merchant_distance",
    "is_fraud"
]

df_ml = df_gold.select(selected_columns)

In [6]:
# Removing null values

df_ml = df_ml.dropna()

print(f"Rows after null handling: {df_ml.count():,}")

Rows after null handling: 1,316,675


In [7]:
# Converting Spark DataFrame to Pandas

pdf = df_ml.toPandas()

In [8]:
# Splitting dataset into train and test

X = pdf.drop("is_fraud", axis=1)
y = pdf["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [9]:
# Training Random Forest model

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Model training completed.")

Model training completed.


In [10]:
# Generating predictions

y_pred = model.predict(X_test)

In [11]:
# Model evaluation

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

Accuracy: 0.9964266048949058
Precision: 0.9033970276008493
Recall: 0.5002939447383892
F1 Score: 0.6439651910707529


In [12]:
# Detailed classification report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    261634
           1       0.90      0.50      0.64      1701

    accuracy                           1.00    263335
   macro avg       0.95      0.75      0.82    263335
weighted avg       1.00      1.00      1.00    263335



In [13]:
# Confusion matrix

conf_matrix = confusion_matrix(y_test, y_pred)

print(conf_matrix)

[[261543     91]
 [   850    851]]


In [14]:
# Feature importance

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance

,Feature,Importance
0,amt,0.392896
7,transaction_hour,0.164508
10,high_amount_flag,0.147468
12,customer_merchant_distance,0.071442
9,customer_age,0.061527
6,unix_time,0.038238
5,merch_long,0.029173
4,merch_lat,0.026877
11,night_transaction_flag,0.025759
1,city_pop,0.017037


In [15]:
# Saving trained model

joblib.dump(model, "/home/jovyan/work/models/fraud_detection_model.pkl")

print("Model successfully saved.")

Model successfully saved.
